# IA supervisionada: classificação de crédito

Este notebook mostra um exemplo simples de IA supervisionada usando `scikit-learn`.

A ideia é prever se um cliente teria crédito `aprovado` ou `negado` com base em renda, idade, score de crédito e dívida atual.

## Problema

Fato: a base tem uma coluna chamada `decisao_credito`, que funciona como resposta correta.

Inferência: se um novo cliente for parecido com clientes aprovados no histórico, o modelo tende a prever aprovação.

Opinião técnica: KNN é uma boa escolha para aula porque é fácil explicar, o modelo compara clientes por similaridade.

Esta célula prepara o ambiente do notebook. Primeiro, ela configura `LOKY_MAX_CPU_COUNT` para evitar um aviso comum do `joblib` no Windows, que não atrapalha o modelo, mas polui a apresentação. Depois, importa `pandas` para manipular tabelas, `plotly` para gráficos e componentes do `scikit-learn` para treino, normalização e avaliação. Esse é o ponto em que você pode explicar que o projeto usa bibliotecas reais de Machine Learning, não uma implementação manual.

In [ ]:
import os

os.environ.setdefault('LOKY_MAX_CPU_COUNT', '1')

import pandas as pd
import plotly.express as px
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MinMaxScaler

pd.set_option('display.max_columns', None)

Esta célula lê uma base fictícia de clientes para o problema de crédito. Cada cliente tem características numéricas, como renda, idade, score de crédito e dívida atual, além da decisão final de crédito. A coluna `decisao_credito` é a resposta conhecida, por isso ela será o alvo do modelo supervisionado. Os dados são pequenos e didáticos, então servem para entender o conceito, não para tomar decisões reais.

In [ ]:
df = pd.read_csv('../data/dados_clientes.csv')

Esta célula valida a base antes de qualquer treinamento. Ela verifica se a base está vazia, se existem valores nulos, se as colunas de entrada são numéricas, se há valores negativos indevidos e se os rótulos são apenas `aprovado` ou `negado`. Também calcula possíveis outliers pelo método IQR, que ajuda a identificar valores muito fora do padrão. Esse passo é importante porque modelo nenhum salva uma análise feita em cima de dados ruins.

In [ ]:
def validar_base_credito(base_credito: pd.DataFrame) -> dict:
    colunas_entrada = ['renda_mensal', 'idade', 'score_credito', 'divida_atual']
    coluna_alvo = 'decisao_credito'
    rotulos_validos = {'aprovado', 'negado'}

    if base_credito.empty:
        raise ValueError('A base de crédito não pode estar vazia.')

    valores_nulos = base_credito.isna().sum().to_dict()
    if any(quantidade > 0 for quantidade in valores_nulos.values()):
        raise ValueError(f'Foram encontrados valores nulos: {valores_nulos}')

    for coluna in colunas_entrada:
        if not pd.api.types.is_numeric_dtype(base_credito[coluna]):
            raise TypeError(f'A coluna {coluna} precisa ser numérica.')

        if (base_credito[coluna] < 0).any():
            raise ValueError(f'A coluna {coluna} possui valor negativo.')

    rotulos_encontrados = set(base_credito[coluna_alvo].unique())
    if not rotulos_encontrados.issubset(rotulos_validos):
        raise ValueError(f'Rótulos inválidos encontrados: {rotulos_encontrados - rotulos_validos}')

    outliers_iqr = {}
    for coluna in colunas_entrada:
        primeiro_quartil = base_credito[coluna].quantile(0.25)
        terceiro_quartil = base_credito[coluna].quantile(0.75)
        intervalo_iqr = terceiro_quartil - primeiro_quartil
        limite_inferior = primeiro_quartil - 1.5 * intervalo_iqr
        limite_superior = terceiro_quartil + 1.5 * intervalo_iqr
        outliers_iqr[coluna] = int(((base_credito[coluna] < limite_inferior) | (base_credito[coluna] > limite_superior)).sum())

    return {
        'quantidade_linhas': len(base_credito),
        'valores_nulos': valores_nulos,
        'rotulos_encontrados': sorted(rotulos_encontrados),
        'outliers_iqr': outliers_iqr,
    }

Esta célula executa a criação da base e chama a função de validação. Em seguida, ela mostra a tabela completa para que seja possível conferir visualmente os registros antes de treinar o modelo. Isso ajuda na apresentação porque você consegue apontar as colunas de entrada e a coluna alvo. É também uma forma simples de mostrar que o fluxo começa pelos dados, não pelo algoritmo.

In [ ]:
resumo_validacao_credito = validar_base_credito(df)

display(df)
resumo_validacao_credito

Esta célula cria um boxplot para visualizar possíveis outliers nas variáveis numéricas da base de crédito. A validação anterior já calcula outliers pelo método IQR, mas o gráfico ajuda a enxergar a distribuição dos valores de forma mais intuitiva. Aqui os pontos individuais foram ocultados para deixar a leitura mais limpa, mostrando apenas caixa, mediana e bigodes de cada variável. Na apresentação, você pode explicar que esse gráfico complementa a validação numérica e ajuda a decidir se algum valor estranho deveria ser investigado antes de treinar o modelo.

In [ ]:
colunas_numericas_credito = ['renda_mensal', 'idade', 'score_credito', 'divida_atual']

dados_boxplot_credito = base_credito.melt(
    value_vars=colunas_numericas_credito,
    var_name='variavel',
    value_name='valor',
)

figura_boxplot_credito = px.box(
    dados_boxplot_credito,
    x='variavel',
    y='valor',
    color='variavel',
    points=False,
    facet_col='variavel',
    facet_col_wrap=2,
    title='Boxplot das variáveis da base de crédito',
)
figura_boxplot_credito.update_yaxes(matches=None, showticklabels=True)
figura_boxplot_credito.update_xaxes(matches=None)
figura_boxplot_credito.update_layout(template='plotly_white', showlegend=False)
figura_boxplot_credito.show()

Esta célula faz a parte principal do aprendizado supervisionado. Ela separa os dados em treino e teste, mantendo a proporção entre `aprovado` e `negado`, cria um `Pipeline` com normalização e KNN, treina o modelo e calcula as métricas. O `MinMaxScaler` é necessário porque KNN compara distâncias, e variáveis em escalas muito diferentes poderiam distorcer o resultado. O uso do `Pipeline` também é uma boa prática, porque mantém pré-processamento e modelo juntos em um fluxo mais organizado.

Precisão, recall e F1-score são métricas usadas para avaliar modelos de classificação, principalmente quando queremos entender melhor os erros do modelo, não só a acurácia.

Precisão mede: quando o modelo disse que era uma classe, quantas vezes ele acertou.

Exemplo: se o modelo disse que 10 clientes seriam `aprovado`, mas só 8 realmente eram aprovados, a precisão foi 8/10 = 80%.

Recall mede: de todos os casos que realmente eram de uma classe, quantos o modelo conseguiu encontrar.

Exemplo: se existiam 10 clientes realmente `aprovado`, mas o modelo encontrou só 7, o recall foi 7/10 = 70%.

F1-score é uma média equilibrada entre precisão e recall.

Ele é útil quando você quer olhar as duas coisas ao mesmo tempo. Se precisão for alta, mas recall for baixo, ou o contrário, o F1-score cai.

Resumo rápido:

| Métrica | Pergunta que responde |
|---|---|
| Precisão | Quando o modelo previu essa classe, ele acertou quanto? |
| Recall | De tudo que era dessa classe, o modelo encontrou quanto? |
| F1-score | Qual o equilíbrio entre precisão e recall? |

No exemplo de crédito:
- precisão baixa em `aprovado` pode significar aprovar clientes que não deveriam ser aprovados, aumentando risco;
- recall baixo em `aprovado` pode significar negar clientes bons, perdendo receita;
- F1-score ajuda a equilibrar esses dois impactos.

In [ ]:
def treinar_modelo_credito(base_credito: pd.DataFrame) -> dict:
    colunas_entrada = ['renda_mensal', 'idade', 'score_credito', 'divida_atual']
    coluna_alvo = 'decisao_credito'

    dados_entrada = base_credito[colunas_entrada]
    alvo_credito = base_credito[coluna_alvo]

    dados_treino, dados_teste, alvo_treino, alvo_teste = train_test_split(
        dados_entrada,
        alvo_credito,
        test_size=0.33,
        random_state=42,
        stratify=alvo_credito,
    )

    modelo_credito = Pipeline(
        steps=[
            ('normalizacao', MinMaxScaler()),
            ('classificador', KNeighborsClassifier(n_neighbors=3)),
        ]
    )
    modelo_credito.fit(dados_treino, alvo_treino)

    previsoes_credito = modelo_credito.predict(dados_teste)
    classes_credito = list(modelo_credito.named_steps['classificador'].classes_)
    matriz_confusao_credito = confusion_matrix(alvo_teste, previsoes_credito, labels=classes_credito)

    resultado_teste = dados_teste.copy()
    resultado_teste['real'] = alvo_teste.to_list()
    resultado_teste['previsto'] = previsoes_credito

    return {
        'modelo_credito': modelo_credito,
        'acuracia_credito': accuracy_score(alvo_teste, previsoes_credito),
        'classes_credito': classes_credito,
        'matriz_confusao_credito': matriz_confusao_credito,
        'resultado_teste': resultado_teste,
        'relatorio_classificacao': classification_report(
            alvo_teste,
            previsoes_credito,
            labels=classes_credito,
            output_dict=True,
            zero_division=0,
        ),
    }


resultado_credito = treinar_modelo_credito(base_credito)

print(f'Acurácia do modelo: {resultado_credito["acuracia_credito"]:.0%}')
display(resultado_credito['resultado_teste'])
display(pd.DataFrame(resultado_credito['relatorio_classificacao']).T.round(2))

Esta célula cria o primeiro gráfico do exemplo supervisionado. O eixo X mostra a renda mensal, o eixo Y mostra o score de crédito, a cor mostra se o cliente foi aprovado ou negado e o tamanho do ponto mostra a dívida atual. O objetivo do gráfico é facilitar a leitura visual dos padrões da base, antes mesmo de olhar as métricas. Na explicação, você pode dizer que bons modelos geralmente começam com uma boa exploração visual dos dados.

In [ ]:
figura_clientes_credito = px.scatter(
    base_credito,
    x='renda_mensal',
    y='score_credito',
    color='decisao_credito',
    size='divida_atual',
    hover_data=['idade', 'divida_atual'],
    title='Clientes por renda, score e decisão de crédito',
    color_discrete_map={'aprovado': '#2f9e44', 'negado': '#d9480f'},
)
figura_clientes_credito.update_layout(template='plotly_white')
figura_clientes_credito.show()

Esta célula cria a matriz de confusão do modelo KNN. A matriz compara a classe real com a classe prevista, então a diagonal principal representa os acertos e os outros campos representariam erros. Mesmo com uma base pequena, esse gráfico é útil porque mostra de forma direta se o modelo confundiu `aprovado` com `negado`. Em uma situação real, essa análise seria essencial porque aprovar errado e negar errado têm impactos de negócio diferentes.

In [ ]:
figura_matriz_confusao = px.imshow(
    resultado_credito['matriz_confusao_credito'],
    x=resultado_credito['classes_credito'],
    y=resultado_credito['classes_credito'],
    text_auto=True,
    labels={'x': 'Previsto', 'y': 'Real', 'color': 'Quantidade'},
    title='Matriz de confusão do KNN',
    color_continuous_scale='Blues',
)
figura_matriz_confusao.update_layout(template='plotly_white')
figura_matriz_confusao.show()

Esta célula simula o uso do modelo depois do treinamento. A função recebe um novo cliente, monta um `DataFrame` com o mesmo formato da base original e usa o modelo para prever a decisão de crédito. Além da classe prevista, ela mostra as probabilidades estimadas para `aprovado` e `negado`, o que deixa a resposta mais interpretável. Esse é o momento de explicar o impacto prático: o modelo poderia apoiar triagem, mas não deveria decidir crédito sozinho em produção.

In [ ]:
def prever_cliente_credito(modelo_credito: Pipeline, dados_novo_cliente: dict) -> dict:
    novo_cliente = pd.DataFrame([dados_novo_cliente])
    classes_credito = list(modelo_credito.named_steps['classificador'].classes_)
    previsao_credito = modelo_credito.predict(novo_cliente)[0]
    probabilidades_credito = modelo_credito.predict_proba(novo_cliente)[0]

    return {
        'previsao_credito': previsao_credito,
        'probabilidades_credito': {
            classe: round(float(probabilidade), 3)
            for classe, probabilidade in zip(classes_credito, probabilidades_credito)
        },
    }


dados_novo_cliente = {
    'renda_mensal': 6000,
    'idade': 34,
    'score_credito': 705,
    'divida_atual': 1900,
}

resultado_novo_cliente = prever_cliente_credito(
    resultado_credito['modelo_credito'],
    dados_novo_cliente,
)

print('Novo cliente analisado:')
print(dados_novo_cliente)
print(f'Previsão: {resultado_novo_cliente["previsao_credito"]}')
print(f'Probabilidades: {resultado_novo_cliente["probabilidades_credito"]}')